In [1]:
import argparse
import os

# # Disable TorchInductor as requested
# os.environ["TORCHINDUCTOR_DISABLE"] = "1"
# os.environ["TORCH_COMPILE"] = "0"
# os.environ["TORCHDYNAMO_DISABLE"] = "1"
# os.environ["DISABLE_TORCH_COMPILE"] = "1"
# os.environ["TRANSFORMERS_NO_COMPILE"] = "1"

import pandas as pd
import torch
from datasets import load_dataset
from tqdm import tqdm
from transformers import (AutoModelForCausalLM,
                          AutoModelForSequenceClassification, AutoTokenizer)

/home/fe/purelku/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
!pip install --upgrade transformers

model_name = "google/gemma-3-1b-pt"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
)

Defaulting to user installation because normal site-packages is not writeable


/home/fe/purelku/.local/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/home/fe/purelku/.local/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [6]:
decoder_layer = model.model.layers[5]
print(f"Decoder layer: {decoder_layer}")

Decoder layer: Gemma3DecoderLayer(
  (self_attn): Gemma3Attention(
    (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
    (k_proj): Linear(in_features=1152, out_features=256, bias=False)
    (v_proj): Linear(in_features=1152, out_features=256, bias=False)
    (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
    (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
    (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
  )
  (mlp): Gemma3MLP(
    (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
    (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
    (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
    (act_fn): PytorchGELUTanh()
  )
  (input_layernorm): Gemma3RMSNorm((1152,), eps=1e-06)
  (post_attention_layernorm): Gemma3RMSNorm((1152,), eps=1e-06)
  (pre_feedforward_layernorm): Gemma3RMSNorm((1152,), eps=1e-06)
  (post_feedforward_layernorm): Gemma3RMSNorm((1152,), eps=1e-06)
)


In [ ]:
print(model.model.layers)
for l in model.model.layers:
    print(l)
   

ModuleList(
  (0-25): 26 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
      (k_proj): Linear(in_features=1152, out_features=256, bias=False)
      (v_proj): Linear(in_features=1152, out_features=256, bias=False)
      (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
      (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
      (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
      (act_fn): PytorchGELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((1152,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((1152,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((1152,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((1152,

In [ ]:
from torch import nn
def inject_identity_layers(model):
    # this assumes your model has a .transformer.h list of blocks
    for i, block in enumerate(model.transformer.h):
        # attach an identity module
        block.identity = nn.Identity()
        # stash the old forward
        original_forward = block.forward
        # define a new forward that applies the identity to the post‐residual output
        def patched_forward(self, x, **kwargs):
            # run the original block (attention + MLP + residual adds inside)
            out = original_forward(x, **kwargs)
            # now pass that result through the identity
            return self.identity(out)
        # bind it
        block.forward = patched_forward.__get__(block, block.__class__)

# inject_identity_layers(model)